# 16.3 隐私保护 (Privacy Protection)

保护训练数据和用户交互中的隐私信息。

本节涵盖：差分隐私、数据脱敏、遗忘学习

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

class DPSGD:
    def __init__(self, model, lr=0.01, noise_multiplier=1.0, max_grad_norm=1.0):
        self.model = model
        self.lr = lr
        self.noise_multiplier = noise_multiplier
        self.max_grad_norm = max_grad_norm

    def step(self):
        total_norm = 0
        for p in self.model.parameters():
            if p.grad is not None:
                total_norm += p.grad.norm() ** 2
        total_norm = total_norm ** 0.5
        clip_coef = self.max_grad_norm / (total_norm + 1e-6)
        for p in self.model.parameters():
            if p.grad is not None:
                p.grad = p.grad * min(clip_coef, 1.0)
                noise = torch.randn_like(p.grad) * self.noise_multiplier * self.max_grad_norm
                p.data -= self.lr * (p.grad + noise)

model = nn.Linear(64, 10)
dp_optimizer = DPSGD(model, noise_multiplier=1.0, max_grad_norm=1.0)

x = torch.randn(4, 64)
y = torch.randint(0, 10, (4,))
loss = F.cross_entropy(model(x), y)
loss.backward()

grad_norm_before = sum(p.grad.norm().item() for p in model.parameters() if p.grad is not None)
dp_optimizer.step()

print('=== Differential Privacy (DP-SGD) ===')
print(f'Noise multiplier: {dp_optimizer.noise_multiplier}')
print(f'Max grad norm: {dp_optimizer.max_grad_norm}')
print(f'\nDP-SGD adds calibrated noise to gradients:')
print(f'  1. Clip gradients to bound sensitivity')
print(f'  2. Add Gaussian noise proportional to sensitivity')
print(f'  3. Guarantees (ε,δ)-differential privacy')
print(f'\nTrade-off: Higher privacy (lower ε) -> more noise -> lower accuracy')
print(f'\n=== Other Privacy Methods ===')
print(f'Data anonymization: remove PII from training data')
print(f'Machine unlearning: make model "forget" specific data')
print(f'Federated learning: train without centralizing data')